# 5 · Fine-tuning Wav2Vec2 XLS-R-300M

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/05_finetune/05_wav2vec2_finetune.ipynb)

**Pipeline stage 5 of 6.** Fine-tune `facebook/wav2vec2-xls-r-300m` for Kölsch
phoneme recognition with a CTC head. Transfer learning from a model pre-trained
on 436k h / 128 languages means a few hours of Kölsch suffice for adaptation.

> Needs a GPU (Colab → Runtime → Change runtime type → GPU).

## Setup

In [ ]:
!pip -q install "transformers>=4.40" datasets evaluate jiwer torchaudio accelerate soundfile librosa
import torch, json, numpy as np
from dataclasses import dataclass
from typing import Dict, List, Union
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "·", device)

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

# On Google Colab: clone the repo once (or mount Drive and point _root at it).
try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/kolsch-tandem.git /content/kolsch-tandem")
except Exception:
    pass

# Repo root = the folder that contains kolsch_paths.py (found from any subfolder).
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, PAGES, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)                       # any remaining relative paths resolve at the root
print("repo root:", ROOT)

## 1 · Load the manifest (audio + phoneme labels)

Join the segment manifest from **Notebook 3** with the `phonetic` labels from
**Notebook 4**. Each row needs an `audio_path` and a `phonetic` string
(`p h | p h`). Then make a speaker-disjoint train/valid/test split.

In [ ]:
from datasets import Dataset
import pandas as pd, os, hashlib, numpy as np

MANIFEST = os.path.join(SEG, "manifest.csv")
assert os.path.exists(MANIFEST), (
    f"{MANIFEST} not found — run Notebook 3 (segmentation) then Notebook 4 "
    "(normalisation) first, so the manifest has audio_path + phonetic columns.")
man = pd.read_csv(MANIFEST)
assert "phonetic" in man.columns, "manifest has no 'phonetic' column — run Notebook 4."
man = man.merge(pd.read_csv(INDEX)[["id","speaker"]], on="id", how="left")

# Speaker-disjoint split when there are enough speakers; otherwise a random
# utterance split so the notebook still runs on tiny / single-speaker data.
def bucket(speaker):
    h = int(hashlib.md5(str(speaker).encode()).hexdigest(), 16) % 10
    return "test" if h < 1 else "valid" if h < 2 else "train"

speakers = man["speaker"].fillna("unknown").unique()
if len(speakers) >= 3:
    man["split"] = man["speaker"].map(bucket)
else:
    rng = np.random.default_rng(0); r = rng.random(len(man))
    man["split"] = np.where(r < 0.8, "train", np.where(r < 0.9, "valid", "test"))
if len(man) > 1 and (man["split"] == "valid").sum() == 0:      # never leave valid empty
    man.loc[man.index[-1], "split"] = "valid"

def to_ds(split):
    sub = man[man["split"] == split][["audio_path", "phonetic"]]
    return Dataset.from_pandas(sub, preserve_index=False)   # keep audio_path as str; decoded in prepare()

train_ds, valid_ds, test_ds = to_ds("train"), to_ds("valid"), to_ds("test")
print("split:", {s: int((man["split"] == s).sum()) for s in ["train","valid","test"]})

## 2 · Build the phoneme vocabulary + processor

In [ ]:
from transformers import (Wav2Vec2PhonemeCTCTokenizer, Wav2Vec2FeatureExtractor,
                          Wav2Vec2Processor)

VOCAB_PATH = os.path.join(MODELS, "vocab.json")
def build_vocab(phonetic_series, path=VOCAB_PATH):
    toks = set()
    for s in phonetic_series:
        toks.update(str(s).split())            # phonemes + the '|' delimiter
    toks.discard("|")
    vocab = {t: i for i, t in enumerate(sorted(toks))}
    vocab["|"] = len(vocab)                     # word delimiter
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)
    json.dump(vocab, open(path, "w"), ensure_ascii=False)
    return vocab

# Labels are ALREADY phonemes (Notebook 4) -> PHONEME tokenizer, do_phonemize=False:
# it maps space-separated IPA phoneme tokens to ids and treats '|' as the word
# delimiter. This is phoneme recognition, not word-level ASR.
vocab = build_vocab(man[man["split"] == "train"]["phonetic"])
tokenizer = Wav2Vec2PhonemeCTCTokenizer(VOCAB_PATH, unk_token="[UNK]",
                pad_token="[PAD]", word_delimiter_token="|",
                phone_delimiter_token=" ", do_phonemize=False)
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000,
                padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
print(f"vocab: {len(vocab)} tokens ->", VOCAB_PATH)

## 3 · Prepare dataset (audio → input_values, phonetic → labels)

In [ ]:
import soundfile as sf
try:
    import librosa
except Exception:
    librosa = None

TARGET_SR = 16000
def load_wav(path):
    wav, sr = sf.read(path)
    if getattr(wav, "ndim", 1) > 1:
        wav = wav.mean(axis=1)                 # to mono
    if sr != TARGET_SR:
        if librosa is None:
            raise RuntimeError(f"{path}: {sr} Hz — pip install librosa to resample to 16 kHz")
        wav = librosa.resample(wav.astype("float32"), orig_sr=sr, target_sr=TARGET_SR)
    return wav

def prepare(batch):
    speech = load_wav(batch["audio_path"])
    batch["input_values"] = processor(speech, sampling_rate=TARGET_SR).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor(text=batch["phonetic"]).input_ids
    return batch

train_ds = train_ds.map(prepare, remove_columns=train_ds.column_names)
valid_ds = valid_ds.map(prepare, remove_columns=valid_ds.column_names)
print("prepared:", len(train_ds), "train /", len(valid_ds), "valid utterances")

## 4 · CTC data collator

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids":   f["labels"]}        for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt")
        # replace padding with -100 so it is ignored by the CTC loss
        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100)
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

## 5 · Metrics — WER and CER (the metrics logged during training)

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    ids = np.argmax(logits, axis=-1)
    labels = pred.label_ids
    labels[labels == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(ids)
    label_str = processor.batch_decode(labels, group_tokens=False)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str),
            "cer": cer_metric.compute(predictions=pred_str, references=label_str)}

## 6 · Model + training configuration

In [ ]:
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.05, hidden_dropout=0.05, feat_proj_dropout=0.05,
    mask_time_prob=0.05, layerdrop=0.05,
    ctc_loss_reduction="mean", ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer))
model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=os.path.join(MODELS, "kolsch_wav2vec2_model"),
    group_by_length=True,
    length_column_name="input_length",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,        # effective batch 16
    per_device_eval_batch_size=2,
    num_train_epochs=150,
    fp16=torch.cuda.is_available(),       # fp16 only on GPU
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=500,
    weight_decay=0.05,
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",          # best checkpoint on validation WER
    greater_is_better=False,
    save_total_limit=2,
)
print("model + training args ready | trainable params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7 · Train, then save model + processor

In [ ]:
import inspect
from transformers import Trainer

# `tokenizer=` was renamed to `processing_class=` in transformers 4.46 and later
# removed. Pick whichever this installed version accepts, so the cell runs on any.
trainer_kwargs = dict(
    model=model, args=args, data_collator=data_collator,
    train_dataset=train_ds, eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
)
_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in _params:
    trainer_kwargs["processing_class"] = processor            # new transformers
elif "tokenizer" in _params:
    trainer_kwargs["tokenizer"] = processor.feature_extractor  # older transformers
trainer = Trainer(**trainer_kwargs)

trainer.train()

out = os.path.join(MODELS, "kolsch_wav2vec2_model")
trainer.save_model(out)
processor.save_pretrained(out)
print("saved ->", out, "| evaluate on the test split in Notebook 6")

## Result

Selecting the best checkpoint on **validation WER** (not loss — CTC validation
loss rebounds) gives ~15.2% val WER / 11.8% val CER, generalising to
**14.63% WER / 11.75% CER** on the held-out test set (see Notebook 6).